In [ ]:
import os
import sys

sys.path.append("../src" if os.path.exists("../src") else "src")
import os
import pandas as pd
from scipy import stats
from utils import REPORT_DIR, get_connection

In [ ]:
conn = get_connection()
df = pd.read_sql_query("SELECT * FROM delivery_master", conn)
done = df[df["status"].isin(["Delivered", "Delayed"])]
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
def show_result(test_name, statistic, p_value):
    """Print the result. If p-value < 0.05 we reject H0 (the result is significant)."""
    print("\n" + test_name)
    print("  statistic =", round(statistic, 4), "| p-value =", round(p_value, 4))
    if p_value < 0.05:
        print("  Result: Reject H0 -> significant")
    else:
        print("  Result: Fail to reject H0 -> not significant")

In [ ]:
q1_q2 = done[done["quarter"].isin([1, 2])]
per_driver = q1_q2.pivot_table(index="driver_id", columns="quarter",
                               values="delivery_hours", aggfunc="mean")
per_driver = per_driver.dropna()
if len(per_driver) > 1:
    t_value, p_value = stats.ttest_rel(per_driver[1], per_driver[2])
    show_result("1. Paired t-test (delivery hours Q1 vs Q2)", t_value, p_value)
    print("  average Q1 =", round(per_driver[1].mean(), 2), "hours | average Q2 =", round(per_driver[2].mean(), 2), "hours")
else:
    print("\n1. Paired t-test skipped: not enough drivers with both Q1 and Q2 data")

In [ ]:
ratings = df["rating"].dropna()
t_value, p_value = stats.ttest_1samp(ratings, 4.0)
show_result("2. One-sample t-test (average rating vs 4.0)", t_value, p_value)
print("  average rating =", round(ratings.mean(), 3))

In [ ]:
pair = done[["distance_km", "delay_hours"]].dropna()
r_value, p_value = stats.pearsonr(pair["distance_km"], pair["delay_hours"])
show_result("3. Pearson correlation (distance vs delay)", r_value, p_value)

In [ ]:
pair = df[["experience", "rating"]].dropna()
rho_value, p_value = stats.spearmanr(pair["experience"], pair["rating"])
show_result("4. Spearman correlation (experience vs rating)", rho_value, p_value)

In [ ]:
delays = done.dropna(subset=["delay_hours"]).copy()
mean_delay = delays["delay_hours"].mean()
std_delay = delays["delay_hours"].std()
delays["z_score"] = (delays["delay_hours"] - mean_delay) / std_delay
outliers = delays[delays["z_score"].abs() > 3]
print("\n5. Z-score: found", len(outliers), "outlier shipments out of", len(delays))

outliers = outliers[["shipment_id", "warehouse_name", "driver_name", "route_type", "delay_hours", "z_score"]]
outliers = outliers.sort_values("z_score", ascending=False)
outliers.to_csv(REPORT_DIR + "delay_outliers.csv", index=False)
print("Outliers saved in the reports folder: delay_outliers.csv")
conn.close()